In [1]:
"""
CarbonTatva AI — Model: XGBoost (Last Year Data)
=================================================
Last-year data: Mar 2025 – Feb 2026
Split:  Train = Mar–Sep 2025  |  Val = Oct–Nov 2025  |  Test = Dec 2025–Feb 2026

Key fixes:
  - Chronological split (train→val→test)
  - Lag features built on FULL data BEFORE filtering
  - Ratio target (price/ma7d) handles distribution shifts within the year
  - No rolling window needed — already 1-year window
"""

import os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
os.makedirs('outputs_xgboost_lastyear', exist_ok=True)

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────
DATA_FILE       = 'final_merged_data.xlsx'
MARKETS         = ['GDAM', 'DAM', 'RTM']
SLOTS_DAY       = 96
LAST_YEAR_START = '2025-03-01'
TRAIN_END       = '2025-09-30'
VAL_END         = '2025-11-30'
SEED            = 42
np.random.seed(SEED)

# ─────────────────────────────────────────────────────────────────
# FEATURE ENGINEERING — build on FULL data first
# ─────────────────────────────────────────────────────────────────
def build_features(df):
    df = df.copy()
    d = pd.to_datetime(df['Delivery Date'], dayfirst=True)
    t = df['Time Period'].str.split('-').str[0]
    df['datetime'] = pd.to_datetime(d.astype(str) + ' ' + t)
    df = df.sort_values('datetime').reset_index(drop=True)

    df.drop(columns=['apparent_temperature (°C)',
                     'sunshine_duration (s)',
                     'precipitation (mm)'], errors='ignore', inplace=True)

    df['hour']    = df['datetime'].dt.hour
    df['dow']     = df['datetime'].dt.dayofweek
    df['month']   = df['datetime'].dt.month
    df['quarter'] = df['datetime'].dt.quarter

    df['hour_sin']  = np.sin(2 * np.pi * df['hour']  / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour']  / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin']   = np.sin(2 * np.pi * df['dow']   / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['dow']   / 7)
    df['is_weekend']   = (df['dow'] >= 5).astype(int)
    df['is_morning']   = ((df['hour'] >= 6)  & (df['hour'] <= 9)).astype(int)
    df['is_evening']   = ((df['hour'] >= 18) & (df['hour'] <= 21)).astype(int)
    df['is_solar_hrs'] = ((df['hour'] >= 8)  & (df['hour'] <= 16)).astype(int)

    try:
        import holidays
        india_h = holidays.India(years=range(2022, 2027))
        df['holiday_flag'] = df['datetime'].dt.date.apply(
            lambda d: 1 if d in india_h else 0)
    except ImportError:
        df['holiday_flag'] = 0

    # AR features on FULL df — so lag values at Mar-2025 use Sep-2024 data
    p = df['Price (Rs./MWh)']
    df['price_lag96']   = p.shift(96)
    df['price_lag192']  = p.shift(192)
    df['price_lag288']  = p.shift(288)
    df['price_lag672']  = p.shift(672)
    df['price_lag1344'] = p.shift(1344)
    df['ma1d']  = p.shift(1).rolling(96).mean()
    df['ma3d']  = p.shift(1).rolling(288).mean()
    df['ma7d']  = p.shift(1).rolling(672).mean()
    df['ma14d'] = p.shift(1).rolling(1344).mean()
    df['ma30d'] = p.shift(1).rolling(2880).mean()
    df['std3d'] = p.shift(1).rolling(288).std()
    df['std7d'] = p.shift(1).rolling(672).std()
    df['price_diff1d'] = p.shift(96)   - p.shift(192)
    df['price_diff1w'] = p.shift(672)  - p.shift(1344)
    df['price_diff2w'] = p.shift(1344) - p.shift(2016)
    df['max_prev_day']   = p.shift(1).rolling(96).max()
    df['min_prev_day']   = p.shift(1).rolling(96).min()
    df['range_prev_day'] = df['max_prev_day'] - df['min_prev_day']

    df['price_level_ratio_7d']  = df['price_lag96'] / (df['ma7d']  + 1e-6)
    df['price_level_ratio_30d'] = df['price_lag96'] / (df['ma30d'] + 1e-6)
    df['rel_lag96']  = df['price_lag96']  / (df['ma7d']  + 1e-6)
    df['rel_lag672'] = df['price_lag672'] / (df['ma30d'] + 1e-6)

    df['temp']    = df['temperature_2m (°C)']
    df['temp_sq'] = df['temp'] ** 2
    df['radiation']            = df['shortwave_radiation (W/m²)']
    df['radiation_x_hour_sin'] = df['radiation'] * df['hour_sin']
    df['radiation_x_is_solar'] = df['radiation'] * df['is_solar_hrs']
    df['wind_speed']  = df['wind_speed_100m (km/h)']
    df['cloud_cover'] = df['cloud_cover (%)']
    df['humidity']    = df['relative_humidity_2m (%)']
    df['rain']        = df['rain (mm)']
    df['temp_x_evening']    = df['temp'] * df['is_evening']
    df['temp_x_morning']    = df['temp'] * df['is_morning']
    df['temp_x_is_weekend'] = df['temp'] * df['is_weekend']

    # Ratio target: price / ma7d  (removes level shifts)
    df['target_ratio'] = p / (df['ma7d'] + 1e-6)
    return df


FEATURES = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'dow_sin', 'dow_cos', 'quarter',
    'is_weekend', 'holiday_flag',
    'is_morning', 'is_evening', 'is_solar_hrs',
    'price_lag96', 'price_lag192', 'price_lag288',
    'price_lag672', 'price_lag1344',
    'ma1d', 'ma3d', 'ma7d', 'ma14d', 'ma30d',
    'std3d', 'std7d',
    'price_diff1d', 'price_diff1w', 'price_diff2w',
    'max_prev_day', 'min_prev_day', 'range_prev_day',
    'price_level_ratio_7d', 'price_level_ratio_30d',
    'rel_lag96', 'rel_lag672',
    'temp', 'temp_sq',
    'radiation', 'radiation_x_hour_sin', 'radiation_x_is_solar',
    'wind_speed', 'cloud_cover', 'humidity', 'rain',
    'temp_x_evening', 'temp_x_morning', 'temp_x_is_weekend',
]

# ─────────────────────────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, label=''):
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    pct  = mae / np.mean(y_true) * 100
    print(f"    {label:22s}  MAE={mae:7.1f}  RMSE={rmse:7.1f}  MAPE={mape:.2f}%  ({pct:.1f}% of mean)")
    return {'mae': mae, 'rmse': rmse, 'mape': mape}


def save_pred_plot(y_true, y_pred, title, path):
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    fig.patch.set_facecolor('#0f1117')
    for ax in axes:
        ax.set_facecolor('#1a1d27'); ax.tick_params(colors='#aaa')
    n = min(800, len(y_true))
    axes[0].plot(y_true[:n], color='#aaa', lw=0.8, label='Actual')
    axes[0].plot(y_pred[:n], color='#e67e22', lw=0.8, label='Predicted')
    axes[0].set_title(f'{title} — Test Set', color='white', fontsize=12)
    axes[0].legend(fontsize=9); axes[0].set_ylabel('Price (Rs./MWh)', color='#aaa')
    n2 = SLOTS_DAY * 5
    axes[1].plot(y_true[:n2], color='#aaa', lw=1.5, label='Actual')
    axes[1].plot(y_pred[:n2], color='#e67e22', lw=1.5, ls='--', label='Predicted')
    axes[1].set_title(f'{title} — First 5 Days', color='white', fontsize=12)
    axes[1].legend(fontsize=9); axes[1].set_ylabel('Price (Rs./MWh)', color='#aaa')
    axes[2].scatter(y_true[:3000], y_pred[:3000], s=2, alpha=0.3, color='#e67e22')
    lim = max(y_true[:3000].max(), y_pred[:3000].max())
    axes[2].plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect')
    axes[2].set_title(f'{title} — Actual vs Predicted', color='white', fontsize=12)
    axes[2].set_xlabel('Actual', color='#aaa'); axes[2].set_ylabel('Predicted', color='#aaa')
    axes[2].legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()


def save_monthly_plot(test_df, test_pred, mkt, path):
    tc = test_df.copy()
    tc['predicted'] = test_pred
    tc['abs_error'] = np.abs(tc['Price (Rs./MWh)'] - test_pred)
    tc['ym'] = tc['datetime'].dt.to_period('M')
    monthly = tc.groupby('ym').agg(
        actual=('Price (Rs./MWh)', 'mean'),
        predicted=('predicted', 'mean'),
        mae=('abs_error', 'mean')).round(1)
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    fig.patch.set_facecolor('#0f1117')
    for ax in axes:
        ax.set_facecolor('#1a1d27'); ax.tick_params(colors='#aaa')
    months = monthly.index.astype(str)
    x = np.arange(len(months)); w = 0.35
    axes[0].bar(x-w/2, monthly['actual'],    w, color='#aaa', alpha=0.8, label='Actual')
    axes[0].bar(x+w/2, monthly['predicted'], w, color='#e67e22', alpha=0.8, label='Predicted')
    axes[0].set_title(f'{mkt} — Monthly Actual vs Predicted', color='white')
    axes[0].set_xticks(x); axes[0].set_xticklabels(months, rotation=30, ha='right')
    axes[0].legend(fontsize=9); axes[0].set_ylabel('Price (Rs./MWh)', color='#aaa')
    axes[1].bar(x, monthly['mae'], color='#e74c3c', alpha=0.85)
    axes[1].set_title(f'{mkt} — Monthly MAE', color='white')
    axes[1].set_xticks(x); axes[1].set_xticklabels(months, rotation=30, ha='right')
    axes[1].set_ylabel('MAE', color='#aaa')
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()
    print(f"\n  Monthly breakdown:\n{monthly.to_string()}")


# ─────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("MODEL: XGBoost (Last Year Data)")
print(f"Period: {LAST_YEAR_START} → Feb 2026")
print(f"Split:  Train≤{TRAIN_END}  |  Val≤{VAL_END}  |  Test>Val")
print("Target: price/MA7d ratio  (handles intra-year level shifts)")
print("=" * 60)

sheets      = pd.read_excel(DATA_FILE, sheet_name=None)
req_feats   = FEATURES + ['target_ratio', 'ma7d', 'Price (Rs./MWh)']
all_metrics = {}

for mkt in MARKETS:
    print(f"\n{'─'*55}\nMarket: {mkt}\n{'─'*55}")

    # Build on FULL data, THEN filter
    df_full = build_features(sheets[mkt])
    df_full = df_full.dropna(subset=req_feats).copy()
    df_full = df_full[np.isfinite(df_full['target_ratio'])].copy()

    df = df_full[df_full['datetime'] >= LAST_YEAR_START].copy()
    print(f"  Rows: {len(df):,}  ({df['datetime'].min().date()} → {df['datetime'].max().date()})")

    train = df[df['datetime'] <= TRAIN_END]
    val   = df[(df['datetime'] > TRAIN_END) & (df['datetime'] <= VAL_END)]
    test  = df[df['datetime'] > VAL_END]
    print(f"  Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
    print(f"  Price mean — Train: {train['Price (Rs./MWh)'].mean():.0f}  Val: {val['Price (Rs./MWh)'].mean():.0f}  Test: {test['Price (Rs./MWh)'].mean():.0f}")

    X_tr = train[FEATURES].values; X_va = val[FEATURES].values; X_te = test[FEATURES].values
    y_tr_ratio = train['target_ratio'].values
    y_va_ratio = val['target_ratio'].values
    ma7d_va = val['ma7d'].values; ma7d_te = test['ma7d'].values
    y_va_abs = val['Price (Rs./MWh)'].values; y_te_abs = test['Price (Rs./MWh)'].values

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)
    X_te_s = scaler.transform(X_te)

    dtrain = xgb.DMatrix(X_tr_s, label=y_tr_ratio, feature_names=FEATURES)
    dval   = xgb.DMatrix(X_va_s, label=y_va_ratio, feature_names=FEATURES)
    dtest  = xgb.DMatrix(X_te_s, feature_names=FEATURES)

    params = {
        'objective': 'reg:squarederror', 'eval_metric': 'mae',
        'max_depth': 6, 'learning_rate': 0.02,
        'subsample': 0.85, 'colsample_bytree': 0.8, 'colsample_bylevel': 0.8,
        'min_child_weight': 20, 'gamma': 0.05,
        'reg_alpha': 0.1, 'reg_lambda': 1.0,
        'tree_method': 'hist', 'max_bin': 256,
        'seed': SEED, 'verbosity': 0,
    }

    print("  Training XGBoost on ratio target...")
    evals_result = {}
    model = xgb.train(
        params, dtrain,
        num_boost_round=3000,
        evals=[(dtrain, 'train'), (dval, 'val')],
        early_stopping_rounds=80,
        evals_result=evals_result,
        verbose_eval=300,
    )
    best_iter = model.best_iteration
    print(f"  Best iteration: {best_iter}")

    val_ratio_pred  = model.predict(dval,  iteration_range=(0, best_iter))
    test_ratio_pred = model.predict(dtest, iteration_range=(0, best_iter))
    val_pred_abs    = np.clip(val_ratio_pred  * ma7d_va, 0, 20000)
    test_pred_abs   = np.clip(test_ratio_pred * ma7d_te, 0, 20000)

    print("  Results:")
    m_val  = compute_metrics(y_va_abs, val_pred_abs,  'Validation')
    m_test = compute_metrics(y_te_abs, test_pred_abs, 'Test')
    all_metrics[mkt] = {'val': m_val, 'test': m_test}

    naive_mae = np.mean(np.abs(y_te_abs - test['price_lag96'].values))
    print(f"    {'Naive lag96 baseline':22s}  MAE={naive_mae:7.1f}  (XGB beats by {(naive_mae-m_test['mae'])/naive_mae*100:.1f}%)")

    # Training curve
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor('#0f1117'); ax.set_facecolor('#1a1d27')
    ax.plot(evals_result['train']['mae'], color='#e74c3c', lw=1, label='Train MAE')
    ax.plot(evals_result['val']['mae'],   color='#27ae60', lw=1, label='Val MAE')
    ax.axvline(best_iter, color='yellow', lw=1.5, ls='--', label=f'Best={best_iter}')
    ax.set_title(f'XGBoost Curve — {mkt}', color='white')
    ax.set_xlabel('Round', color='#aaa'); ax.set_ylabel('MAE (ratio)', color='#aaa')
    ax.legend(fontsize=9); ax.tick_params(colors='#aaa')
    plt.tight_layout()
    plt.savefig(f'outputs_xgboost_lastyear/xgb_loss_{mkt}.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()

    # Feature importance
    imp = model.get_score(importance_type='gain')
    if imp:
        names = list(imp.keys()); vals = list(imp.values())
        idx = np.argsort(vals)[-25:]
        fig, ax = plt.subplots(figsize=(11, 9))
        fig.patch.set_facecolor('#0f1117'); ax.set_facecolor('#1a1d27')
        ax.barh([names[i] for i in idx], [vals[i] for i in idx], color='#e67e22', alpha=0.85)
        ax.set_title(f'Feature Importance — {mkt}', color='white'); ax.tick_params(colors='#aaa')
        ax.set_xlabel('Gain', color='#aaa')
        plt.tight_layout()
        plt.savefig(f'outputs_xgboost_lastyear/xgb_importance_{mkt}.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
        plt.close()

    save_pred_plot(y_te_abs, test_pred_abs, f'XGBoost — {mkt}',
                   f'outputs_xgboost_lastyear/xgb_pred_{mkt}.png')
    save_monthly_plot(test, test_pred_abs, mkt,
                      f'outputs_xgboost_lastyear/xgb_monthly_{mkt}.png')

    model.save_model(f'outputs_xgboost_lastyear/xgb_{mkt}.json')
    with open(f'outputs_xgboost_lastyear/xgb_scaler_{mkt}.pkl', 'wb') as f:
        pickle.dump({'scaler': scaler, 'features': FEATURES}, f)
    print(f"  Saved: outputs_xgboost_lastyear/xgb_{mkt}.json")

print(f"\n{'='*60}\nSUMMARY — XGBoost (Last Year)\n{'='*60}")
rows = []
for mkt, res in all_metrics.items():
    for split, m in res.items():
        rows.append({'Market': mkt, 'Split': split,
                     'MAE': round(m['mae'],1), 'RMSE': round(m['rmse'],1),
                     'MAPE%': round(m['mape'],2)})
summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
summary_df.to_csv('outputs_xgboost_lastyear/metrics.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('XGBoost — Test MAE (Last Year)', color='white', fontsize=14)
colors = {'GDAM':'#e74c3c','DAM':'#2980b9','RTM':'#27ae60'}
for ax, mkt in zip(axes, MARKETS):
    ax.set_facecolor('#1a1d27')
    if mkt in all_metrics:
        mae = all_metrics[mkt]['test']['mae']
        ax.bar([mkt], [mae], color=colors[mkt], alpha=0.85)
        ax.text(0, mae+5, f'Rs.{mae:.0f}', ha='center', color='white', fontsize=12, fontweight='bold')
    ax.set_title(mkt, color='white'); ax.set_ylabel('MAE (Rs./MWh)', color='#aaa'); ax.tick_params(colors='#aaa')
plt.tight_layout()
plt.savefig('outputs_xgboost_lastyear/summary_chart.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.close()
print("\nDone! Check outputs_xgboost_lastyear/")


MODEL: XGBoost (Last Year Data)
Period: 2025-03-01 → Feb 2026
Split:  Train≤2025-09-30  |  Val≤2025-11-30  |  Test>Val
Target: price/MA7d ratio  (handles intra-year level shifts)

───────────────────────────────────────────────────────
Market: GDAM
───────────────────────────────────────────────────────
  Rows: 35,040  (2025-03-01 → 2026-02-28)
  Train: 20,449 | Val: 5,856 | Test: 8,735
  Price mean — Train: 4657  Val: 3579  Test: 4256
  Training XGBoost on ratio target...
[0]	train-mae:0.43906	val-mae:0.38937
[186]	train-mae:0.08988	val-mae:0.16062
  Best iteration: 106
  Results:
    Validation              MAE=  541.0  RMSE=  816.8  MAPE=23.75%  (15.1% of mean)
    Test                    MAE=  550.2  RMSE=  840.1  MAPE=12.62%  (12.9% of mean)
    Naive lag96 baseline    MAE=  432.5  (XGB beats by -27.2%)

  Monthly breakdown:
         actual  predicted    mae
ym                               
2025-11  3679.5     3872.9  380.3
2025-12  4461.6     4527.2  552.8
2026-01  4385.8     45